### **Question 01**

- Using the following URL, you can get the weather data of any place, by specifing the place latitude and longitude or its name:

https://rapidapi.com/worldapi/api/open-weather13/

- Signup, login then subscribe to `(Open Weather)` then select `3-hour Forecast 5 days` API (be sure to choose the basic free plan)
- Follow the provided instuction to be able to fetch data from their website.
- Fetch data for the following cities:

    `cities=['New York', 'San Francisco', 'Washington DC', 'Buffalo']`
    
    where (You will need to use the coordinates in order to fetch the data):
    
    `coordinates = {'New York':{'Latitude':40.712776,'Longitude': -74.005974}, 'San Francisco': {'Latitude':37.774929,'Longitude': -122.419418}, 'Washington DC':{'Latitude':38.907192,'Longitude': -77.036873},'Buffalo':{'Latitude':42.886448,'Longitude': -78.878372}}`
    
    
- For each city make a Dataframe that looks as follows:

|| dt |	humidity |	pressure |	temp_min |	feels_like |
|---|---|---|---|---|---|
| 0 |	1635984000 |	54.55 |	1024.79 |	282.73 |	285.93 |
| 1 |	1636070400 |	69.28 |	1025.25 |	284.13 |	282.33 |
|2 |	1636156800 |	73.19 |	1017.72 |	285.40 |	282.71 |
|3 |	1636243200 |	62.23 |	1018.62 |	283.74 |	280.72 |
|4 |	1636329600 |	57.89 |	1020.54 |	281.01 |	281.73 |
|5 |	1636416000 |	61.58 |	1021.16 |	280.58 |	282.13 |
|6 |	1636502400 |	60.40 |	1017.86 |	281.93 |	282.22 |
|7 |	1636588800 |	61.60 |	1019.52 |	281.80 |	283.47 |
|.. |	.......... |	..... |	....... |	...... |	.... |
|29 |	1638489600 |	73.70 |	1014.98 |	279.25 |	280.06 |

- Save each dataFrame (without the index) in a file that has the same name of the city with extension `csv` (for example: `New York.csv`). 

In [1]:
import os
import requests
import pandas as pd


API_KEY = "256f9c79b1msh83ee95a40f518f0p1d5fd1jsnb70b576811d7"

BASE_URL = "https://open-weather13.p.rapidapi.com/fivedaysforcast"

cities = ['New York', 'San Francisco', 'Washington DC', 'Buffalo']
coordinates = {
    'New York': {'Latitude': 40.712776, 'Longitude': -74.005974},
    'San Francisco': {'Latitude': 37.774929, 'Longitude': -122.419418},
    'Washington DC': {'Latitude': 38.907192, 'Longitude': -77.036873},
    'Buffalo': {'Latitude': 42.886448, 'Longitude': -78.878372}
}

if not API_KEY:
    raise ValueError("Set API_KEY first.")

headers = {
    "x-rapidapi-key": API_KEY,
    "x-rapidapi-host": "open-weather13.p.rapidapi.com"
}

city_dfs = {}

for city in cities:
    lat = coordinates[city]["Latitude"]
    lon = coordinates[city]["Longitude"]

    querystring = {
        "latitude": lat, 
        "longitude": lon, 
        "lang": "EN"
    }

    response = requests.get(
        BASE_URL,
        headers=headers,
        params=querystring,
        timeout=30
    )
    
    if response.status_code != 200:
        print(f"Failed to fetch {city}: Status Code {response.status_code}")
        print(response.text)
        continue # Skip to the next city if this one fails
        
    response.raise_for_status()

    data = response.json()
    rows = []
    
    for item in data.get("list", []):
        main = item.get("main", {})
        rows.append({
            "dt": item.get("dt"),
            "humidity": main.get("humidity"),
            "pressure": main.get("pressure"),
            "temp_min": main.get("temp_min"),
            "feels_like": main.get("feels_like"),
        })

    df_city = pd.DataFrame(rows, columns=["dt", "humidity", "pressure", "temp_min", "feels_like"])
    city_dfs[city] = df_city
    df_city.to_csv(f"{city}.csv", index=False)

    print(f"{city}: saved {len(df_city)} rows to {city}.csv")

New York: saved 40 rows to New York.csv
San Francisco: saved 40 rows to San Francisco.csv
Washington DC: saved 40 rows to Washington DC.csv
Buffalo: saved 40 rows to Buffalo.csv


In [74]:
# Your code here

### **Question 02**
- Combine all previous dataframes in one with additional column that's the city name. This should look as follows:


| |dt| 	humidity |	pressure |	temp_min |	feels_like |	city_name |
|---|---|---|---|---|---|---|
| 0 |	1635984000 |	54.55 |	1024.79 |	282.73 |	285.93 |	New York |
| 1 | 	1636070400 | 69.28 |	1025.25 |	284.13 |	282.33 |	New York |
| 2 |	1636156800 |	73.19 |	1017.72 |	285.40 |	282.71 |	New York |
| ... |	... |	... |	... |	... |	... |	... |
| 28 |	1638403200 |	77.29 |	1016.95 |	278.76 |	280.81 |	Washington DC |
| 29 |	1638489600 |	70.03 |	1017.47 |	279.89 |	289.94 |	Washington DC |

**Hint:**
You can add column to a dataFrame df using: `df['col_name']=value`

- Set the index in the new data frame to be the city_name
- Save this data frame as (all.csv)

In [2]:
bigdf = pd.concat(city_dfs, names=["city_name"]).reset_index(level=0)
bigdf = bigdf.set_index("city_name")
bigdf.to_csv("all.csv")
bigdf.head()


,dt,humidity,pressure,temp_min,feels_like
city_name,,,,,
New York,1772247600,84,1022,274.11,273.13
New York,1772258400,87,1021,274.05,272.59
New York,1772269200,91,1020,273.94,272.15
New York,1772280000,93,1020,273.96,272.26
New York,1772290800,84,1020,277.31,275.15


In [3]:
# Your code here

In [4]:
# Your code here

### **Question 03**
- Using the data you had from previous question, find the following:
    - The city that has the highest feels_like
    - The city that has the lowest pressure
    - The highest humidity in each city

In [9]:
city_highest_feels_like = bigdf["feels_like"].idxmax()

city_highest_feels_like = bigdf["feels_like"].idxmax()
max_feels_like = bigdf["feels_like"].max()


In [10]:
city_lowest_pressure = bigdf["pressure"].idxmin()
min_pressure = bigdf["pressure"].min()


In [11]:

highest_humidity_each_city = bigdf.groupby(level="city_name")["humidity"].max()

print("City with highest feels_like:", city_highest_feels_like, "| value:", max_feels_like)
print("City with lowest pressure:", city_lowest_pressure, "| value:", min_pressure)
print("\nHighest humidity in each city:")
print(highest_humidity_each_city)


City with highest feels_like: San Francisco | value: 294.43
City with lowest pressure: Buffalo | value: 1009

Highest humidity in each city:
city_name
Buffalo          99
New York         99
San Francisco    92
Washington DC    99
Name: humidity, dtype: int64


### **Question 04**

- use `pd.to_datetime` to convert `dt` into `datetime` then extract `day` and `hour`
- Then, for each city, find:
    - The average feels_like per day.
    - The time of day (hour) when humidity is typically highest.

In [81]:
# ### **Question 04**

# - use `pd.to_datetime` to convert `dt` into `datetime` then extract `day` and `hour`
# - Then, for each city, find:
#     - The average feels_like per day.
#     - The time of day (hour) when humidity is typically highest.
bigdf["datetime"] = pd.to_datetime(bigdf["dt"], unit="s")
bigdf["day"] = bigdf["datetime"].dt.day
bigdf["hour"] = bigdf["datetime"].dt.hour


avg_feels_like_per_day = bigdf.groupby(["city_name", "day"])["feels_like"].mean()

hour_with_highest_humidity = bigdf.groupby(["city_name", "hour"])["humidity"].mean().groupby(level=0).idxmax()

avg_feels_like_per_day = bigdf.groupby([bigdf.index, "day"])["feels_like"].mean()
print(avg_feels_like_per_day)
print("\nHour with highest humidity for each city:")
print(hour_with_highest_humidity)

city_name      day
Buffalo        1      262.64125
               2      261.29375
               3      267.15875
               4      271.82625
               28     272.44625
New York       1      273.16125
               2      262.68875
               3      268.72750
               4      274.47750
               28     275.12375
San Francisco  1      287.05500
               2      286.18750
               3      285.73000
               4      285.41500
               28     290.62125
Washington DC  1      281.40625
               2      270.38875
               3      269.88625
               4      279.32500
               28     280.17750
Name: feels_like, dtype: float64

Hour with highest humidity for each city:
city_name
Buffalo                (Buffalo, 12)
New York               (New York, 6)
San Francisco    (San Francisco, 15)
Washington DC    (Washington DC, 12)
Name: humidity, dtype: object


In [82]:

bigdf["datetime"] = pd.to_datetime(bigdf["dt"], unit="s")
bigdf["day"] = bigdf["datetime"].dt.day
bigdf["hour"] = bigdf["datetime"].dt.hour

# Average feels_like per day per city
avg_feels_like_per_day = bigdf.groupby([bigdf.index, "day"])["feels_like"].mean()

# Hour when humidity is typically highest per city
hour_with_highest_humidity = (
    bigdf.groupby([bigdf.index, "hour"])["humidity"]
    .mean()
    .groupby(level=0)
    .idxmax()
    .apply(lambda x: x[1])  # keep hour only
)

print("Average feels_like per day:")
print(avg_feels_like_per_day)
print("\nHour with highest typical humidity:")
print(hour_with_highest_humidity)

Average feels_like per day:
city_name      day
Buffalo        1      262.64125
               2      261.29375
               3      267.15875
               4      271.82625
               28     272.44625
New York       1      273.16125
               2      262.68875
               3      268.72750
               4      274.47750
               28     275.12375
San Francisco  1      287.05500
               2      286.18750
               3      285.73000
               4      285.41500
               28     290.62125
Washington DC  1      281.40625
               2      270.38875
               3      269.88625
               4      279.32500
               28     280.17750
Name: feels_like, dtype: float64

Hour with highest typical humidity:
city_name
Buffalo          12
New York          6
San Francisco    15
Washington DC    12
Name: humidity, dtype: int64


### **Question 05**

- Create a pivot table that shows the average temp_min and humidity for each city across days.
- Group data by city_name and compute: Mean, Min, Max, Std for feels_like.
- Sort results to find the top 2 cities with the highest average feels_like.



In [83]:
# ### **Question 05**

# - Create a pivot table that shows the average temp_min and humidity for each city across days.
# - Group data by city_name and compute: Mean, Min, Max, Std for feels_like.
# - Sort results to find the top 2 cities with the highest average feels_like.


pivot_table = bigdf.pivot_table(index="city_name", columns="day", values=["temp_min", "humidity"], aggfunc="mean")  

feels_like_stats = bigdf.groupby("city_name")["feels_like"].agg(["mean", "min", "max", "std"])

top_cities_feels_like = feels_like_stats.sort_values(by="mean", ascending=False).head(2)
print("Pivot table:")
print(pivot_table)
print("\nTop 2 cities by average feels_like:")
print(top_cities_feels_like)

Pivot table:
              humidity                                   temp_min             \
day                 1       2       3       4       28         1          2    
city_name                                                                      
Buffalo         77.250  66.250  77.250  97.625  70.875  268.09750  264.05500   
New York        78.125  43.000  57.125  96.000  79.625  276.23375  269.05625   
San Francisco   84.750  83.125  83.250  75.875  75.250  287.36625  286.61375   
Washington DC   71.500  39.875  65.000  89.375  64.625  282.79250  273.86875   

                                                
day                   3          4          28  
city_name                                       
Buffalo        269.37750  273.53500  276.07875  
New York       272.18375  276.99625  276.39000  
San Francisco  286.19625  286.08500  290.41375  
Washington DC  273.11000  281.58750  280.88250  

Top 2 cities by average feels_like:
                    mean     min     max      

In [84]:
# ...existing code...
# Pivot: average temp_min and humidity by city across days
pivot_table = bigdf.pivot_table(
    index=bigdf.index,
    columns="day",
    values=["temp_min", "humidity"],
    aggfunc="mean"
)

# feels_like stats per city
feels_like_stats = bigdf.groupby(level="city_name")["feels_like"].agg(["mean", "min", "max", "std"])

# Top 2 cities by highest average feels_like
top_cities_feels_like = feels_like_stats.sort_values(by="mean", ascending=False).head(2)

print("Pivot table:")
print(pivot_table)
print("\nFeels_like stats:")
print(feels_like_stats)
print("\nTop 2 cities by average feels_like:")
print(top_cities_feels_like)


Pivot table:
              humidity                                   temp_min             \
day                 1       2       3       4       28         1          2    
city_name                                                                      
Buffalo         77.250  66.250  77.250  97.625  70.875  268.09750  264.05500   
New York        78.125  43.000  57.125  96.000  79.625  276.23375  269.05625   
San Francisco   84.750  83.125  83.250  75.875  75.250  287.36625  286.61375   
Washington DC   71.500  39.875  65.000  89.375  64.625  282.79250  273.86875   

                                                
day                   3          4          28  
city_name                                       
Buffalo        269.37750  273.53500  276.07875  
New York       272.18375  276.99625  276.39000  
San Francisco  286.19625  286.08500  290.41375  
Washington DC  273.11000  281.58750  280.88250  

Feels_like stats:
                    mean     min     max       std
city_name    

### **Question 06**
- Which city tends to be the most humid overall?
- Which city has lowest air pressure stability (highest std)?
- Which city is most comfortable overall (based on feels_like averages)?
- What correlations exist between humidity and feels_like? (hint: use df.corr())

In [85]:
# ### **Question 06**
# - Which city tends to be the most humid overall?
# - Which city has lowest air pressure stability (highest std)?
# - Which city is most comfortable overall (based on feels_like averages)?
# - What correlations exist between humidity and feels_like? (hint: use df.corr())
most_humid_city = bigdf.groupby("city_name")["humidity"].mean().idxmax()
lowest_pressure_stability_city = bigdf.groupby("city_name")["pressure"].std().idxmax()
most_comfortable_city = bigdf.groupby("city_name")["feels_like"].mean().idxmax()
correlation_humidity_feels_like = bigdf[["humidity", "feels_like"]].corr().loc["humidity", "feels_like"]
print("Most humid city overall:", most_humid_city)
print("Lowest pressure stability city:", lowest_pressure_stability_city)
print("Most comfortable city overall:", most_comfortable_city)
print("Humidity vs feels_like correlation:", correlation_humidity_feels_like)

Most humid city overall: San Francisco
Lowest pressure stability city: Buffalo
Most comfortable city overall: San Francisco
Humidity vs feels_like correlation: 0.20309731172020748


In [ ]:
most_humid_city = bigdf.groupby(level="city_name")["humidity"].mean().idxmax()
lowest_pressure_stability_city = bigdf.groupby(level="city_name")["pressure"].std().idxmax()
most_comfortable_city = bigdf.groupby(level="city_name")["feels_like"].mean().idxmax()
correlation_humidity_feels_like = bigdf[["humidity", "feels_like"]].corr().loc["humidity", "feels_like"]

print("Most humid city overall:", most_humid_city)
print("Lowest pressure stability city:", lowest_pressure_stability_city)
print("Most comfortable city overall:", most_comfortable_city)
print("Humidity vs feels_like correlation:", correlation_humidity_feels_like)

Most humid city overall: San Francisco
Lowest pressure stability city: Buffalo
Most comfortable city overall: San Francisco
Humidity vs feels_like correlation: 0.20309731172020748
